## Match Zip Codes to Counties

1. zillow.csv will require a county for each zip code entry, ensure no missing leading zero's
2. Gather range by state (ex. NY has specific, CA has specific zipcodes)
2. full center join via zip and county for validity
3. manual CV of random samples


In [11]:
import pandas as pd

zillow_df = pd.read_csv('zillow_corrected_address.csv')
prop_df = pd.read_csv('2025_prop.csv')


In [ ]:
zillow_df['zipcode'] = zillow_df['zipcode'].astype(str).str.zfill(5)

# verify
print(zillow_df['zipcode'].head(10))
print(f"Error Check: {(zillow_df['zipcode'].str.len() < 5).sum()}")

0    92373
1    91423
2    91606
3    92054
4    90042
5    93722
6    91784
7    92506
8    95008
9    93308
Name: zipcode, dtype: object
Any still 4 digits: 0


In [ ]:
import pgeocode

# start
n = pgeocode.Nominatim('us')

# zfill ensures 5 numbers
# + county bc other sheet uses that
def zip_to_county(zipcode):
    result = n.query_postal_code(str(zipcode).zfill(5))
    if result is not None and pd.notna(result['county_name']):
        return result['county_name'] + ' County'
    return None

zillow_df['county_lookup'] = zillow_df['zipcode'].apply(zip_to_county)

In [14]:
display(zillow_df.head())

,Unnamed: 0,zpid,address,price,beds,baths,area_sqft,latitude,longitude,status,...,detail_url,has_open_house,is_featured,street_add,city,state_zipcode,state_code,zipcode,state_name,county_lookup
0,0,17264897,"979 Kevin Ave, Redlands, CA 92373",447000,3.0,2.0,1300.0,34.040520,-117.186195,House for sale,...,https://www.zillow.com/homedetails/979-Kevin-A...,False,False,979 Kevin Ave,Redlands,CA 92373,CA,92373,California,San Bernardino County
1,1,20021372,"13114 Addison St, Sherman Oaks, CA 91423",2795000,4.0,5.0,2900.0,34.160885,-118.418770,House for sale,...,https://www.zillow.com/homedetails/13114-Addis...,True,False,13114 Addison St,Sherman Oaks,CA 91423,CA,91423,California,Los Angeles County
2,2,20009320,"6032 Goodland Ave, North Hollywood, CA 91606",1718000,4.0,3.0,3025.0,34.180450,-118.411280,House for sale,...,https://www.zillow.com/homedetails/6032-Goodla...,False,False,6032 Goodland Ave,North Hollywood,CA 91606,CA,91606,California,Los Angeles County
3,3,460204882,"3325 Tonopah St, Oceanside, CA 92054",899999,3.0,3.0,1682.0,33.214260,-117.341500,Coming soon,...,https://www.zillow.com/homedetails/3325-Tonopa...,False,False,3325 Tonopah St,Oceanside,CA 92054,CA,92054,California,San Diego County
4,4,20769150,"963 Pine Grove Ave, Los Angeles, CA 90042",995000,2.0,2.0,1271.0,34.132890,-118.186220,House for sale,...,https://www.zillow.com/homedetails/963-Pine-Gr...,False,False,963 Pine Grove Ave,Los Angeles,CA 90042,CA,90042,California,Los Angeles County


## Correction of State Tax Table

1. Change values of None to 0
2. Expand out decimal places (10.9% currently shows as 11% in excel, unsure how it will show in a df)
3. Define metric, highest marginal tax rate?  Tax Rate based on average home price?  Tax Rate based on average income?
4. Flatten table